[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/05_machine_learning_fundamentals/90_exercises.ipynb)

# Bài tập — Chương 05: ML Fundamentals (L0 → L7)

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/05_machine_learning_fundamentals

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold, KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from ml_course.checks import check_close, check_equal, check_less, check_true

rng = np.random.default_rng(0)

## L0 — Nhớ khái niệm

**Câu 1.** Với mỗi bài toán, cho biết loại (regression / classification / clustering / anomaly detection / reinforcement learning): (a) dự đoán số ngày nằm viện; (b) phát hiện giao dịch bất thường khi không có nhãn; (c) phân khúc khách hàng; (d) robot học đi; (e) email có phải spam.

✍️ Trả lời:

## L1 — Giải thích bằng lời

**Câu 2.** Giải thích bias và variance cho một người không học ML, dùng ví dụ bắn cung.

✍️ Trả lời:

**Câu 3.** Vì sao điểm số của model "tốt nhất" được chọn trên tập test bị lạc quan giả tạo?

✍️ Trả lời:

## L2 — Tính tay

**Câu 4.** Con người đạt lỗi 3%. Model: lỗi train 4%, lỗi validation 14%. Tính avoidable bias và variance; vấn đề chính là gì?

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
avoidable_bias = None
variance = None
main_problem = None
### END CODE HERE ###

In [ ]:
check_equal((avoidable_bias, variance), (1, 10), "avoidable bias = 1%, variance = 10%")
check_equal(main_problem, "variance", "vấn đề chính")

## L3 — Code

**Bài 5.** Viết `bias_variance(degree, n_datasets=100, n_points=25, noise=0.3, seed=0)` ước lượng **bias²** và **variance** (trung bình trên lưới `grid` ở vùng $[-2.5, 2.5]$) của hồi quy đa thức bậc `degree` cho hàm thật $f(x) = \sin(1.5x)$, $x \sim U(-3, 3)$. Trả về tuple `(bias2, variance)`.

In [ ]:
grid = np.linspace(-2.5, 2.5, 100)   # vùng bên trong — tránh ngoại suy ở mép


def bias_variance(degree, n_datasets=100, n_points=40, noise=0.3, seed=0):
    r = np.random.default_rng(seed)
    preds = []
    for _ in range(n_datasets):
        x = r.uniform(-3, 3, n_points)
        y = np.sin(1.5 * x) + r.normal(0, noise, n_points)
        model = make_pipeline(PolynomialFeatures(degree), StandardScaler(), LinearRegression())
        ### START CODE HERE ### (≈ 1 dòng)
        pass
        ### END CODE HERE ###
    preds = np.array(preds)
    ### START CODE HERE ### (≈ 2 dòng)
    bias2 = None
    variance = None
    ### END CODE HERE ###
    return bias2, variance


for d in [1, 5, 12]:
    print(d, np.round(bias_variance(d), 4))

In [ ]:
b1, v1 = bias_variance(1)
b5, v5 = bias_variance(5)
b12, v12 = bias_variance(12)
check_true(b1 > 10 * b5, "bậc 1 có bias lớn hơn nhiều so với bậc 5")
check_true(v12 > 5 * v5, "bậc 12 có variance lớn hơn nhiều so với bậc 5")

**Bài 6.** Viết `group_split(groups, test_frac, seed)` trả về `(train_idx, test_idx)` sao cho **không nhóm nào** xuất hiện ở cả hai tập: chọn ngẫu nhiên khoảng `test_frac` số **nhóm** cho test.

In [ ]:
def group_split(groups, test_frac=0.2, seed=0):
    groups = np.asarray(groups)
    r = np.random.default_rng(seed)
    ### START CODE HERE ### (≈ 4 dòng)
    unique = None
    test_groups = None
    test_mask = None
    train_idx, test_idx = None, None
    ### END CODE HERE ###
    return train_idx, test_idx

In [ ]:
g = np.repeat(np.arange(30), 5)
tr, te = group_split(g, 0.2)
check_true(not set(g[tr]) & set(g[te]), "không nhóm nào nằm ở cả train và test")
check_equal(len(set(g[te])), 6, "20% số nhóm vào test")
check_equal(len(tr) + len(te), len(g), "không mất mẫu")

## L4 — Debug

**Bài 7.** Đoạn code sau báo cáo accuracy ~0.9 trên dữ liệu **ngẫu nhiên**. Tìm lỗi và sửa (kết quả đúng phải quanh 0.5).

In [ ]:
X_rand = rng.normal(size=(60, 2000))
y_rand = rng.integers(0, 2, 60)

# 🐛 code của đồng nghiệp
scaled = StandardScaler().fit_transform(X_rand)
selected = SelectKBest(f_classif, k=15).fit_transform(scaled, y_rand)
buggy_score = cross_val_score(LogisticRegression(), selected, y_rand, cv=5).mean()

### START CODE HERE ### (≈ 2 dòng)
pipe = None
fixed_score = None
### END CODE HERE ###
print(f"buggy = {buggy_score:.2f} | fixed = {fixed_score:.2f}")

In [ ]:
check_true(buggy_score > 0.75, "tái hiện được leakage")
check_less(abs(fixed_score - 0.5), 0.2, "sau khi sửa, accuracy ≈ 0.5")

## L5 — Phân tích

**Bài 8.** Dùng `sklearn.model_selection.learning_curve` cho hai model trên dữ liệu `moons` (2 000 mẫu, `noise=0.3`): (a) Logistic Regression tuyến tính; (b) `KNeighborsClassifier(1)`. Vẽ learning curve (tỉ lệ lỗi) và chẩn đoán từng model. Thêm dữ liệu có giúp model nào?

In [ ]:
from sklearn.model_selection import learning_curve
from sklearn.neighbors import KNeighborsClassifier

from ml_course.datasets import make_classification_2d

Xm, ym = make_classification_2d("moons", n=2000, noise=0.3)
curves = {}
### START CODE HERE ### (≈ 4 dòng)
pass
### END CODE HERE ###
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, (s, tr_err, va_err)) in zip(axes, curves.items()):
    ax.plot(s, tr_err, "o-", label="train")
    ax.plot(s, va_err, "s-", label="validation")
    ax.set_title(name)
    ax.legend()
plt.show()

In [ ]:
_, tr_log, va_log = curves["logistic"]
_, tr_knn, va_knn = curves["knn1"]
check_true(abs(tr_log[-1] - va_log[-1]) < 0.03 and va_log[-1] > 0.1, "logistic: hai đường hội tụ ở lỗi cao → high bias")
check_true(tr_knn[-1] < 0.01 and va_knn[-1] - tr_knn[-1] > 0.1, "KNN k=1: lỗi train ≈ 0, khoảng cách lớn → high variance")

✍️ Chẩn đoán:

## L6 — Thực tế

**Bài 9.** Dữ liệu cảm biến: 40 người, mỗi người 25 mẫu, nhãn là **hoạt động** (không phụ thuộc người) nhưng mỗi người có "phong cách" riêng. So sánh accuracy của `KNeighborsClassifier(5)` với `KFold` ngẫu nhiên và `GroupKFold` theo người. Cách nào phản ánh đúng hiệu năng khi triển khai cho **người dùng mới**?

In [ ]:
n_people, per = 40, 25
person = np.repeat(np.arange(n_people), per)
activity = rng.integers(0, 3, n_people * per)
style = rng.normal(0, 2.0, size=(n_people, 6))[person]               # khác biệt giữa người (lớn)
signal = np.eye(3)[activity] @ rng.normal(0, 1.0, size=(3, 6))       # tín hiệu hoạt động (nhỏ hơn)
X_sensor = style + signal + rng.normal(0, 0.5, size=style.shape)
### START CODE HERE ### (≈ 3 dòng)
knn = None
acc_kfold = None
acc_group = None
### END CODE HERE ###
print(f"KFold: {acc_kfold:.3f} | GroupKFold: {acc_group:.3f}")

In [ ]:
check_true(acc_kfold > acc_group + 0.05, "KFold ngẫu nhiên lạc quan hơn GroupKFold")

## L7 — Nâng cao

**Bài 10.** Cài đặt **adversarial validation**: hàm `adversarial_auc(A, B)` trả về ROC-AUC (5-fold `cross_val_predict` với `RandomForestClassifier(n_estimators=100, random_state=0)`) của classifier phân biệt hàng từ `A` (nhãn 0) và `B` (nhãn 1). Kiểm tra: hai mẫu cùng phân phối → AUC ≈ 0.5; dịch chuyển trung bình một feature → AUC cao.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_predict


def adversarial_auc(A, B):
    ### START CODE HERE ### (≈ 5 dòng)
    X = None
    is_b = None
    clf = None
    proba = None
    auc = None
    ### END CODE HERE ###
    return auc


A = rng.normal(size=(800, 4))
B_same = rng.normal(size=(800, 4))
B_shift = rng.normal(size=(800, 4)) + np.array([0, 1.2, 0, 0])   # dịch 1.2σ ở một feature
print(adversarial_auc(A, B_same), adversarial_auc(A, B_shift))

In [ ]:
check_less(abs(adversarial_auc(A, B_same) - 0.5), 0.06, "cùng phân phối → AUC ≈ 0.5")
check_true(adversarial_auc(A, B_shift) > 0.7, "có shift → AUC cao")